# פייז 7 — PINO

בדיקות בלבד — אין תא שמפעיל אימון.

## פתיחה ב־Colab
העלו מחברת זו ל־Colab. בתחילת runtime חדש תא ההכנה יבקש את `spno-colab-source.zip` המצורף, אלא אם הקוד המעודכן כבר נמצא ב־`PROJECT_ROOT`. חיבור Drive נעשה דרך ממשק Colab הרגיל.

הגדירו את נתיב תוצרי פייז 6. הנתיב המקורי יכול להיות ב־Drive או בדיסק המקומי של הריצה שטרם נסגרה. אפשר להעתיק את התוצרים ל־Drive באמצעות `COPY_TO`; המקור אינו נמחק. חשוב להשלים את ההעתקה לפני סגירת runtime שבו התוצרים נמצאים רק תחת `/content`.

המחברת קוראת משקולות קיימות. `B-post` הוא A עם projection; אין לו אימון נפרד. checkpoints ישנים אינם מכילים optimizer ולכן אינם נקודת המשך מדויקת לאימון שנקטע. כל אימון חדש דרך הקטלוג שומר התקדמות בסוף כל epoch.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
PROJECT_ROOT = Path(os.environ.get("SPNO_PROJECT_ROOT", "/content/spno-colab" if IN_COLAB else str(Path.cwd())))
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    if not (PROJECT_ROOT / "src/spno/workflow.py").exists():
        # Upload the supplied spno-colab-source.zip once per fresh runtime.
        import zipfile
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload the single spno-colab-source.zip bundle")
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                if not (PROJECT_ROOT / member.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive member")
            archive.extractall(PROJECT_ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT) + "[dirichlet,notebooks]"])
if not (PROJECT_ROOT / "src/spno/workflow.py").is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the updated spno source directory")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
OPTIONS = json.loads(os.environ.get("SPNO_OPTIONS", "{}"))
from IPython.display import display, HTML, Image


In [ ]:
# הנתיב הוא לתיקיית artifacts של standalone, ולא לתיקיית הגרפים.
SOURCE_ROOT = Path(os.environ.get("SPNO_SOURCE_ROOT", "/content/drive/MyDrive/spno/phase6-standalone-artifacts"))
OUTPUT_ROOT = Path(os.environ.get("SPNO_OUTPUT_ROOT", "/content/drive/MyDrive/spno/workflow" if IN_COLAB else str(PROJECT_ROOT / "results/colab-workflow")))
CACHE_ROOT = Path("/content/spno-data-cache") if IN_COLAB else None
# אם המקור עדיין בדיסק הזמני של הריצה הישנה: שנו את SOURCE_ROOT לנתיבו.
# COPY_TO מעתיק ל-Drive ובודק שלמות; None משאיר את המקור במקומו.
COPY_TO = None
# קובץ אופציונלי עם {"data": {...}, "train": {...}} מהאימון המקורי.
SOURCE_CONFIG = os.environ.get("SPNO_SOURCE_CONFIG")
# אפשר להצהיר כאן על TrainConfig המקורי אם הדוח אינו מכיל את תקציב האימון.
# לדוגמה, רק אם זו אכן הפקודה שרצה: {"epochs": 80, "batch_size": 256, "learning_rate": 1e-3, "patience": 8}
SOURCE_TRAIN_CONFIG = None
SEEDS = OPTIONS.get("seeds")  # None = כל ה-seeds שהתגלו בפייז 6
DEVICE = OPTIONS.get("device", "auto")
ALLOW_BUDGET_BOUND = OPTIONS.get("allow_budget_bound", False)

# שינוי הניסוי החדש בלבד; אינו משנה את הצהרת פרוטוקול המקור.
TRAIN_OVERRIDES = OPTIONS.get("train_overrides", {})


In [ ]:
from dataclasses import asdict, replace
from spno.artifacts import atomic_json, copy_standalone
from spno.config import DataConfig
from spno.train import TrainConfig
from spno.workflow import Workflow
from spno.experiments import pick_device
from spno.phase_workflow import evaluate_phase

if COPY_TO is not None:
    SOURCE_ROOT = copy_standalone(SOURCE_ROOT, Path(COPY_TO))
source_config = json.loads(Path(SOURCE_CONFIG).read_text()) if SOURCE_CONFIG else {}
if SOURCE_TRAIN_CONFIG is not None:
    source_config["train"] = SOURCE_TRAIN_CONFIG
workflow = Workflow(
    SOURCE_ROOT, OUTPUT_ROOT, cache_root=CACHE_ROOT,
    data_config=DataConfig(**source_config["data"]) if "data" in source_config else None,
    train_config=TrainConfig(**source_config["train"]) if "train" in source_config else None,
)
SEEDS = workflow.seeds if SEEDS is None else SEEDS
DEVICE = pick_device(DEVICE)
if workflow.train_config is not None:
    atomic_json(OUTPUT_ROOT / "source-config.json", {
        "data": asdict(workflow.data_config), "train": asdict(workflow.train_config),
    })
import html
rows = workflow.inventory()
headers = ["model", "seed", "converged", "best epoch", "history", "protocol", "architecture"]
body = []
for row in rows:
    values = [row["name"], row["seed"], row["converged"], row["metadata"]["best_epoch"],
              "available" if row["history"] else "not recorded", row["protocol_source"] or "UNKNOWN",
              row["metadata"]["architecture"]]
    body.append("<tr>" + "".join("<td>" + html.escape(str(v)) + "</td>" for v in values) + "</tr>")
display(HTML("<table><tr>" + "".join("<th>" + h + "</th>" for h in headers) + "</tr>" + "".join(body) + "</table>"))
print("Seeds:", SEEDS, "Device:", DEVICE)
print("Original training protocol:", workflow.train_config or "UNKNOWN — complete SOURCE_TRAIN_CONFIG before preparing matched experiments")

if TRAIN_OVERRIDES and workflow.train_config is None:
    raise ValueError("Declare the original training protocol before choosing overrides")
REQUESTED_TRAIN_CONFIG = replace(workflow.train_config, **TRAIN_OVERRIDES) if TRAIN_OVERRIDES else None


## הניסוי
מציגים את כל סוויפ lambda. הביקורת lambda=0 משתמשת ב־A רק אם תנאי האימון תואמים. שאר הערכים הם מודלים שאומנו בנפרד, מאותו seed. ה־Crank–Nicolson residual משמר מסה בעצמו: יש לציין זאת לצד התוצאות.

checkpoint חסר עוצר את הבדיקה עם מזהה הניסוי. חזרו למחברת האימון כדי להכינו. `ALLOW_BUDGET_BOUND=True` מתיר בדיקה מסומנת כחקרנית; הוא אינו משנה את מצב ההתכנסות השמור.

In [ ]:
result = evaluate_phase(
    workflow, 7, seeds=SEEDS, device=DEVICE,
    allow_budget_bound=ALLOW_BUDGET_BOUND, train_config=REQUESTED_TRAIN_CONFIG,
    lambdas=OPTIONS.get("lambdas", [0.0, 0.01, 0.1, 1.0, 10.0]),
)


In [ ]:
print("Saved:", result["output"])
print("Exploratory / budget-bound:", result["exploratory"])
for path in sorted((Path(result["output"]) / "plots").glob("*.png")):
    display(Image(filename=str(path)))
